# Loan Approval & Credit Risk Analysis
### Final Data Analytics Capstone Project

**Domain:** Finance / Banking
**Dataset:** Synthetic loan applications (3,215 records)
**Author:** Yogesh Kumar Mourya

This notebook covers Task 3 (Data Preparation) and Task 4 (Exploratory Data Analysis) of the capstone assignment.

## Task 3: Dataset Selection & Data Preparation

### 3.1 Load the dataset and inspect structure

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("loan_applications_raw.csv")
print("Shape:", df.shape)
df.head()

Shape: (3215, 16)


,application_id,age,gender,marital_status,education,employment_type,annual_income,credit_score,existing_loans_count,loan_amount,loan_term_months,loan_purpose,region,application_date,approval_status,default_flag
0,LN100670,63,Female,Married,Graduate,Salaried,67300.0,620.0,4,10600.0,24,Education,West,2024-05-24,Approved,0.0
1,LN102335,45,Male,Married,Post Graduate,Self-Employed,21200.0,722.0,0,9000.0,60,Auto,North,2024-05-10,Approved,0.0
2,LN102503,25,Male,Married,Not Graduate,Self-Employed,45700.0,746.0,1,11800.0,12,Education,South,2023-10-18,Approved,0.0
3,LN102586,37,Male,Married,Not Graduate,Self-Employed,43200.0,693.0,0,10400.0,24,Personal,East,2023-12-21,Rejected,NaN
4,LN100819,65,female,Single,Post Graduate,Salaried,41000.0,718.0,4,21100.0,48,Business,East,2023-11-02,Rejected,NaN


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3215 entries, 0 to 3214
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   application_id        3215 non-null   str    
 1   age                   3215 non-null   int64  
 2   gender                3215 non-null   str    
 3   marital_status        3167 non-null   str    
 4   education             3215 non-null   str    
 5   employment_type       2972 non-null   str    
 6   annual_income         3118 non-null   float64
 7   credit_score          3150 non-null   float64
 8   existing_loans_count  3215 non-null   int64  
 9   loan_amount           3215 non-null   float64
 10  loan_term_months      3215 non-null   int64  
 11  loan_purpose          3215 non-null   str    
 12  region                3183 non-null   str    
 13  application_date      3215 non-null   str    
 14  approval_status       3215 non-null   str    
 15  default_flag          1324 non-n

### 3.2 Identify missing values

In [3]:
df.isna().sum().sort_values(ascending=False)

default_flag            1891
employment_type          243
annual_income             97
credit_score              65
marital_status            48
region                    32
gender                     0
age                        0
education                  0
application_id             0
existing_loans_count       0
loan_amount                0
loan_purpose               0
loan_term_months           0
application_date           0
approval_status            0
dtype: int64

**Observation:** `default_flag` has ~1,890 missing values by design — it's only applicable to
`Approved` applications (a rejected loan was never disbursed, so it can't default). The remaining
columns (`employment_type`, `annual_income`, `credit_score`, `marital_status`, `region`) have genuine
missing data that needs imputation.

### 3.3 Identify duplicate and invalid records

In [4]:
print("Duplicate rows:", df.duplicated().sum())
print("\nInvalid ages (<0):", (df["age"] < 0).sum())
print("Invalid loan terms (>120 months):", (df["loan_term_months"] > 120).sum())
print("\nInconsistent gender values:", df["gender"].unique())
print("Inconsistent employment_type values:", df["employment_type"].unique())

Duplicate rows: 15

Invalid ages (<0): 3
Invalid loan terms (>120 months): 3

Inconsistent gender values: <StringArray>
['Female', 'Male', 'female', 'F', 'male', 'M']
Length: 6, dtype: str
Inconsistent employment_type values: <StringArray>
[      'Salaried',  'Self-Employed', 'Business Owner',              nan,
      'Salaried ',  'self-employed']
Length: 6, dtype: str


### 3.4 Data cleaning

In [5]:
# Remove exact duplicate rows
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows")

Removed 15 duplicate rows


In [6]:
# Standardize inconsistent categorical text
df["gender"] = df["gender"].str.strip().str.lower().map({
    "male": "Male", "m": "Male", "female": "Female", "f": "Female"
}).fillna(df["gender"])

df["employment_type"] = df["employment_type"].str.strip().str.title()
df["employment_type"] = df["employment_type"].replace({"Self-employed": "Self-Employed"})

print(df["gender"].unique())
print(df["employment_type"].unique())

<StringArray>
['Female', 'Male']
Length: 2, dtype: str
<StringArray>
['Salaried', 'Self-Employed', 'Business Owner', nan]
Length: 4, dtype: str


In [7]:
# Fix impossible / invalid values by converting them to missing, then imputing
df.loc[df["age"] < 0, "age"] = np.nan
df.loc[df["loan_term_months"] > 120, "loan_term_months"] = np.nan

### 3.5 Handle missing values

In [8]:
# Numeric columns -> median imputation (robust to outliers/skew)
for col in ["annual_income", "credit_score", "age", "loan_term_months"]:
    df[col] = df[col].fillna(df[col].median())

# Categorical columns -> mode imputation
for col in ["marital_status", "employment_type", "region"]:
    df[col] = df[col].fillna(df[col].mode()[0])

# default_flag: NaN is structurally meaningful (rejected loans), so we leave it as a
# nullable integer rather than imputing a value that doesn't apply
df["default_flag"] = df["default_flag"].astype("Int64")

df.isna().sum()

application_id             0
age                        0
gender                     0
marital_status             0
education                  0
employment_type            0
annual_income              0
credit_score               0
existing_loans_count       0
loan_amount                0
loan_term_months           0
loan_purpose               0
region                     0
application_date           0
approval_status            0
default_flag            1887
dtype: int64

### 3.6 Correct data types

In [9]:
df["age"] = df["age"].astype(int)
df["loan_term_months"] = df["loan_term_months"].astype(int)
df["application_date"] = pd.to_datetime(df["application_date"])

df.dtypes

application_id                     str
age                              int64
gender                             str
marital_status                     str
education                          str
employment_type                    str
annual_income                  float64
credit_score                   float64
existing_loans_count             int64
loan_amount                    float64
loan_term_months                 int64
loan_purpose                       str
region                             str
application_date        datetime64[us]
approval_status                    str
default_flag                     Int64
dtype: object

### 3.7 Save the cleaned dataset

In [10]:
df.to_csv("loan_applications_cleaned.csv", index=False)
print("Cleaned dataset saved:", df.shape)

Cleaned dataset saved: (3200, 16)


---
## Task 4: Exploratory Data Analysis (EDA)

### 4.1 Descriptive statistics

In [11]:
df[["age", "annual_income", "credit_score", "loan_amount",
    "loan_term_months", "existing_loans_count"]].describe()

,age,annual_income,credit_score,loan_amount,loan_term_months,existing_loans_count
count,3200.000000,3200.000000,3200.000000,3200.000000,3200.000000,3200.0000
mean,43.035938,68173.062500,654.402187,27003.781250,39.783750,2.0200
std,12.988403,61758.757178,79.823483,28308.894791,16.974342,1.4299
min,21.000000,12000.000000,354.000000,3000.000000,12.000000,0.0000
25%,32.000000,48200.000000,599.000000,14900.000000,24.000000,1.0000
50%,43.000000,63300.000000,655.000000,23150.000000,36.000000,2.0000
75%,54.000000,78200.000000,708.000000,33200.000000,48.000000,3.0000
max,65.000000,944900.000000,850.000000,576100.000000,84.000000,4.0000


**Observations:**
- Median annual income is ~₹63,300, but the mean (~₹68,200) is pulled higher by a long right tail — a sign of income outliers.
- Credit scores are roughly centered around 650-655, consistent with a normal-ish distribution.
- Loan amounts range from ₹3,000 to over ₹576,000, with a median of ~₹23,150.

### 4.2 Correlation analysis

In [12]:
numeric_cols = ["age", "annual_income", "credit_score", "loan_amount",
                 "loan_term_months", "existing_loans_count"]
corr = df[numeric_cols].corr()
corr

,age,annual_income,credit_score,loan_amount,loan_term_months,existing_loans_count
age,1.000000,-0.002070,0.004628,0.002710,-0.029269,-0.015288
annual_income,-0.002070,1.000000,0.020564,0.885743,0.009466,0.015435
credit_score,0.004628,0.020564,1.000000,0.023462,0.010158,0.006442
loan_amount,0.002710,0.885743,0.023462,1.000000,0.024008,0.025890
loan_term_months,-0.029269,0.009466,0.010158,0.024008,1.000000,0.007545
existing_loans_count,-0.015288,0.015435,0.006442,0.025890,0.007545,1.000000


**Key finding:** `annual_income` and `loan_amount` are strongly correlated (**r ≈ 0.89**) — makes
sense, since loan amounts were sized as a proportion of income. Age, credit score, loan term, and
existing loan count show negligible correlation with each other, meaning risk isn't simply a function
of any single one of these variables in isolation.

### 4.3 Outlier detection (IQR method)

In [13]:
def iqr_outliers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return series[(series < lower) | (series > upper)]

income_outliers = iqr_outliers(df["annual_income"])
loan_outliers = iqr_outliers(df["loan_amount"])

print(f"Income outliers: {len(income_outliers)} records ({len(income_outliers)/len(df)*100:.1f}%)")
print(f"Loan amount outliers: {len(loan_outliers)} records ({len(loan_outliers)/len(df)*100:.1f}%)")

Income outliers: 35 records (1.1%)
Loan amount outliers: 52 records (1.6%)


**Note:** These outliers represent genuine high-net-worth applicants rather than data errors — they're
retained in the dataset but flagged, since removing them would understate the real range of loan
applicants a bank sees.

### 4.4 Key patterns: approval and default rates

In [14]:
approval_rate = (df["approval_status"] == "Approved").mean() * 100
print(f"Overall approval rate: {approval_rate:.1f}%")

approved = df[df["approval_status"] == "Approved"]
default_rate = approved["default_flag"].mean() * 100
print(f"Default rate among approved loans: {default_rate:.1f}%")

Overall approval rate: 41.0%
Default rate among approved loans: 12.3%


In [15]:
approval_by_purpose = df.groupby("loan_purpose")["approval_status"].apply(
    lambda x: (x == "Approved").mean() * 100
).sort_values(ascending=False)
approval_by_purpose

loan_purpose
Business              43.253968
Debt Consolidation    42.428571
Education             41.709845
Medical               41.040462
Home Improvement      40.219780
Auto                  39.506173
Personal              37.151703
Name: approval_status, dtype: float64

In [16]:
default_by_credit_band = approved.assign(
    credit_band=pd.cut(approved["credit_score"], bins=[300, 580, 670, 740, 800, 850],
                        labels=["Poor", "Fair", "Good", "Very Good", "Excellent"])
).groupby("credit_band", observed=True)["default_flag"].mean() * 100
default_by_credit_band

credit_band
Poor         16.393443
Fair         13.888889
Good         13.900415
Very Good     7.971014
Excellent     7.142857
Name: default_flag, dtype: Float64

**Key findings summary:**
1. Overall approval rate is ~41% — the model is fairly conservative.
2. Default rate among approved loans is ~12.3% overall, but it isn't evenly spread — it falls steadily
   as credit score improves, from ~16% in the "Poor" band to ~7% in "Excellent."
3. `Business` and `Debt Consolidation` loans have the highest approval rates (~42-43%); `Personal` loans
   have the lowest (~37%).
4. Annual income is the strongest single driver of loan amount size, but not of approval or default —
   credit score is the better risk signal.

These findings carry into Task 5 (Visualizations), Task 6 (Power BI Dashboard), and Task 7 (Business
Insights & Recommendations).